# 1. Environment and Version Checks

Verify that all required packages are installed and environment variables are configured.
This cell prints Python, NumPy, Pandas, and Optuna versions, and checks for `MONGO_URI` and `OPTUNA_STORAGE` env vars.

In [4]:
import sys, os, subprocess

# Ensure pmm_lab is importable
PMM_DIR = "/quants-lab/research_notebooks/market_lab/pmm_dynamic"
if PMM_DIR not in sys.path:
    sys.path.insert(0, PMM_DIR)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", PMM_DIR, "--quiet"],
                      stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

import numpy as np
import pandas as pd
import optuna
import pmm_lab

print(f"Python version  : {sys.version.split()[0]}")
print(f"NumPy version   : {np.__version__}")
print(f"Pandas version  : {pd.__version__}")
print(f"Optuna version  : {optuna.__version__}")
print(f"pmm_lab version : {pmm_lab.__version__}")
print()
MONGO_URI = os.getenv("MONGO_URI", "")
OPTUNA_STORAGE = os.getenv("OPTUNA_STORAGE", "")
print(f"MONGO_URI       : {'SET' if MONGO_URI else 'NOT SET (will skip MongoDB cells)'}")
print(f"OPTUNA_STORAGE  : {'SET' if OPTUNA_STORAGE else 'NOT SET (will use SQLite)'}")

Python version  : 3.12.12
NumPy version   : 2.2.6
Pandas version  : 3.0.1
Optuna version  : 4.7.0
pmm_lab version : 0.1.0

MONGO_URI       : SET
OPTUNA_STORAGE  : SET


# 2. Connect to MongoDB

Ping the MongoDB instance and report the total candle count.
If `MONGO_URI` is not configured this section will be skipped gracefully.

In [2]:
try:
    from pymongo import MongoClient

    if not MONGO_URI:
        raise EnvironmentError("MONGO_URI is not set -- skipping MongoDB connection.")

    client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")
    print("MongoDB ping: OK")

    db = client.get_default_database()
    candle_count = db["candles"].estimated_document_count()
    print(f"Estimated candle documents in collection: {candle_count:,}")
except Exception as exc:
    print(f"[SKIP] MongoDB connection failed or unavailable: {exc}")
    client = None

MongoDB ping: OK
Estimated candle documents in collection: 2,332,110


# 3. Dataset Discovery

List available connector / trading-pair / interval combinations stored in the database.
This uses the `CandleLoader` helper to enumerate combos filtered by `quote_asset='USDT'`.

In [3]:
try:
    from pmm_lab.data.mongo import MongoCandleLoader

    if not MONGO_URI:
        raise EnvironmentError("MONGO_URI not set")

    loader = MongoCandleLoader()
    combos = loader.list_combos(quote_asset="USDT")
    print(f"Available USDT-quote combos: {len(combos)}")
    combos_df = pd.DataFrame(combos)
    display(combos_df)
except Exception as exc:
    print(f"[SKIP] Dataset discovery failed: {exc}")
    combos_df = pd.DataFrame()

Available USDT-quote combos: 66


,connector,trading_pair,interval,count,first_ts,last_ts
0,mexc,BTC-USDT,15m,17658,1756943100,1772834400
1,mexc,BTC-USDT,1d,184,1756944000,1772755200
2,mexc,BTC-USDT,1h,4415,1756944000,1772834400
3,mexc,BTC-USDT,1m,48885,1769902200,1772835240
4,mexc,BTC-USDT,4h,1104,1756944000,1772827200
...,...,...,...,...,...,...
61,nonkyc,XMR-USDT,15m,17948,1756682100,1772834400
62,nonkyc,XMR-USDT,1d,505,1729209600,1772755200
63,nonkyc,XMR-USDT,1h,4597,1756288800,1772834400
64,nonkyc,XMR-USDT,4h,1525,1750881600,1772827200


# 4. Candle Extraction and Validation

Load candles for a selected connector, trading pair, and interval.
Run `validate_candles()` to produce an audit summary (gap count, duplicates, null fraction, etc.).

In [4]:
# --- Configuration: change these to match your dataset ---
CONNECTOR = "nonkyc"
TRADING_PAIR = "SAL-USDT"
INTERVAL = "5m"
BAR_INTERVAL_SECONDS = 300  # must match INTERVAL
# ----------------------------------------------------------

In [5]:
try:
    from pmm_lab.data.mongo import MongoCandleLoader
    from pmm_lab.data.candles import validate_candles
    from pmm_lab.config.params import DataQuery
    from pmm_lab.data.hashing import hash_candles

    if not MONGO_URI:
        raise EnvironmentError("MONGO_URI not set")

    loader = MongoCandleLoader()
    query = DataQuery(connector=CONNECTOR, trading_pair=TRADING_PAIR, interval=INTERVAL)
    candles = loader.load_range(query)
    print(f"Loaded {len(candles):,} candles  [{candles['timestamp'][0]} .. {candles['timestamp'][-1]}]")

    audit = validate_candles(candles, interval=INTERVAL, strict=True)
    print(f"\n--- Candle Audit Summary ---")
    print(f"  Total rows       : {audit.total_rows:,}")
    print(f"  Expected rows    : {audit.expected_rows:,}")
    print(f"  Missing rows     : {audit.missing_rows:,}")
    print(f"  Duplicates       : {audit.duplicate_count}")
    print(f"  OHLC violations  : {audit.ohlc_violations}")
    print(f"  Volume=0 bars    : {audit.volume_zero_count} ({audit.volume_zero_fraction:.2%})")
    print(f"  Strict pass      : {audit.passed_strict}")
    if audit.failure_reasons:
        for r in audit.failure_reasons:
            print(f"    FAIL: {r}")

    DATASET_HASH = hash_candles(candles)
    print(f"\n  Dataset hash     : {DATASET_HASH[:16]}...")
except Exception as exc:
    print(f"[SKIP] Candle extraction/validation failed: {exc}")
    candles = None
    DATASET_HASH = None

Loaded 53,143 candles  [1756892400 .. 1772835000]

--- Candle Audit Summary ---
  Total rows       : 53,143
  Expected rows    : 53,143
  Missing rows     : 0
  Duplicates       : 0
  OHLC violations  : 0
  Volume=0 bars    : 1041 (1.96%)
  Strict pass      : True

  Dataset hash     : c3f47aaeb70a0c77...


# 5. Forward-Fill Analysis

Report the number and fraction of candles with zero volume, which are a proxy for
forward-filled (stale) bars.  A more advanced forward-fill detection pipeline is
deferred to v2.

In [6]:
if candles is not None:
    zero_vol = np.sum(candles["volume"] == 0)
    total = len(candles)
    print(f"Volume=0 candles: {zero_vol} / {total} ({zero_vol/total:.2%})")
    print()
    print("Note: Forward-fill detection is deferred to v2.")
    print("Volume=0 bars are not filled by the simulator (volume > 0 required for fills).")
else:
    print("[SKIP] No candle data loaded.")

Volume=0 candles: 1041 / 53143 (1.96%)

Note: Forward-fill detection is deferred to v2.
Volume=0 bars are not filled by the simulator (volume > 0 required for fills).


# 6. Timestamp Semantics

Define whether the candle timestamp represents the **open** or **close** of the bar.
Most exchange APIs use open-time semantics, so that is the default.

In [7]:
timestamp_mode = "open"  # options: "open" | "close"

print(f"timestamp_mode = '{timestamp_mode}'")
print()
print("Explanation:")
print("  'open'  -- each candle's timestamp marks the START of the bar interval.")
print("             This is the default for Binance, Kucoin, Gate, etc.")
print("  'close' -- each candle's timestamp marks the END of the bar interval.")
print("             Use this if your data source follows close-time semantics.")

timestamp_mode = 'open'

Explanation:
  'open'  -- each candle's timestamp marks the START of the bar interval.
             This is the default for Binance, Kucoin, Gate, etc.
  'close' -- each candle's timestamp marks the END of the bar interval.
             Use this if your data source follows close-time semantics.


# 7. PMM Dynamic Feature Parity

Compute the PMM Dynamic strategy features (reference price, spread multiplier, etc.)
on the loaded candle data and display a few sample bars to verify parity with the
live strategy implementation.

In [8]:
try:
    from pmm_lab.features.pmm_dynamic_features import compute_pmm_dynamic_features, PMMDynamicConfig
    from pmm_lab.features.alignment import align_features

    if candles is None:
        raise ValueError("No candle data loaded.")

    feat_config = PMMDynamicConfig()
    features = compute_pmm_dynamic_features(candles, feat_config)
    aligned = align_features(features, timestamp_mode=timestamp_mode)

    print(f"Warmup ends at bar {aligned.warmup_end} (features valid from here)")
    print(f"\nLast 5 bars:")
    for i in range(-5, 0):
        idx = len(candles) + i
        print(f"  Bar {idx}: ref_price={aligned.reference_price[idx]:.2f}, "
              f"spread_mult={aligned.spread_multiplier[idx]:.6f}, "
              f"natr={aligned.natr[idx]:.6f}")
except Exception as exc:
    print(f"[SKIP] Feature computation failed: {exc}")
    aligned = None

Warmup ends at bar 52 (features valid from here)

Last 5 bars:
  Bar 53138: ref_price=0.03, spread_mult=0.008374, natr=0.008374
  Bar 53139: ref_price=0.03, spread_mult=0.008305, natr=0.008305
  Bar 53140: ref_price=0.03, spread_mult=0.007959, natr=0.007959
  Bar 53141: ref_price=0.03, spread_mult=0.007895, natr=0.007895
  Bar 53142: ref_price=0.03, spread_mult=0.007843, natr=0.007843


# 8. Baseline CPU Backtest

Run a single backtest with default parameters to establish a performance baseline.
Uses `SimConfig` for configuration and `CandleSimRunner` for execution.

In [9]:
try:
    from pmm_lab.sim.executor_model import SimConfig
    from pmm_lab.sim.runner import CandleSimRunner
    from pmm_lab.config.params import PairRules, FeeConfig
    from pmm_lab.config.exchange_rules import load_exchange_rules, resolve_pair_rules

    if candles is None:
        raise ValueError("No candle data loaded.")

    # Load exchange rules for the target pair
    rules_db = load_exchange_rules()
    pair_rules = resolve_pair_rules(rules_db, CONNECTOR, TRADING_PAIR)

    # Default baseline config
    baseline_config = SimConfig(
        buy_spreads=[1.0, 2.0, 4.0],
        sell_spreads=[1.0, 2.0, 4.0],
        buy_amounts_pct=[0.33, 0.34, 0.33],
        sell_amounts_pct=[0.33, 0.34, 0.33],
        total_amount_quote=100.0,
    )

    runner = CandleSimRunner(baseline_config, pair_rules)
    baseline_result = runner.run(candles)

    print("=== Baseline Backtest Summary ===")
    print(f"  Trades completed : {len(baseline_result.trades)}")
    print(f"  Orders placed    : {baseline_result.n_orders_placed}")
    print(f"  Orders filled    : {baseline_result.n_orders_filled}")
    print(f"  Orders rejected  : {baseline_result.n_orders_rejected}")
    print(f"  Market exits     : {baseline_result.n_market_exits}")
    print(f"  Final equity     : {baseline_result.equity_curve[-1]:.2f}")
    print(f"  PnL (quote)      : {baseline_result.equity_curve[-1] - baseline_config.total_amount_quote:.2f}")
except Exception as exc:
    print(f"[SKIP] Baseline backtest failed: {exc}")
    baseline_result = None
    baseline_config = None

=== Baseline Backtest Summary ===
  Trades completed : 12796
  Orders placed    : 16440
  Orders filled    : 12796
  Orders rejected  : 0
  Market exits     : 6884
  Final equity     : -687.27
  PnL (quote)      : -787.27


# 9. Metrics and Objective Decomposition

Compute the full metrics dictionary from the simulation result, then evaluate
`objective_v1` and display the individual components that contribute to the
composite objective score.

In [10]:
try:
    from pmm_lab.metrics.metrics import compute_metrics
    from pmm_lab.objective.objective import objective_v1

    if baseline_result is None:
        raise ValueError("No baseline result available.")

    metrics = compute_metrics(
        baseline_result,
        initial_equity=baseline_config.total_amount_quote,
        candles=candles,
        bar_interval_seconds=BAR_INTERVAL_SECONDS,
    )

    obj = objective_v1(metrics)

    print(f"=== Metrics ===")
    print(f"  PnL %          : {metrics.pnl_pct:.4f}")
    print(f"  Sharpe         : {metrics.sharpe:.4f}")
    print(f"  Max Drawdown % : {metrics.max_drawdown_pct:.4f}")
    print(f"  Trade count    : {metrics.trade_count}")
    print(f"  Profit factor  : {metrics.profit_factor:.4f}")
    print(f"  Total fees     : {metrics.total_fees_quote:.4f}")
    print(f"  Fee drag %     : {metrics.fee_drag_pct:.4f}")
    print()
    print(f"=== Objective Decomposition ===")
    print(f"  Raw score      : {obj.raw_score:.6f}")
    print(f"  PnL component  : {obj.pnl_component:.6f}")
    print(f"  Sharpe comp.   : {obj.sharpe_component:.6f}")
    print(f"  Drawdown comp. : {obj.drawdown_component:.6f}")
    print(f"  Fee drag comp. : {obj.fee_drag_component:.6f}")
    print(f"  Inventory comp.: {obj.inventory_component:.6f}")
    print(f"  Trade penalty  : {obj.trade_count_penalty:.6f}")
    print(f"  Rejected?      : {obj.is_rejected}")
except Exception as exc:
    print(f"[SKIP] Metrics/objective failed: {exc}")
    metrics = None
    obj = None

=== Metrics ===
  PnL %          : -787.2653
  Sharpe         : 0.6645
  Max Drawdown % : 789.7404
  Trade count    : 12796
  Profit factor  : 0.4015
  Total fees     : 203.0815
  Fee drag %     : 203.0815

=== Objective Decomposition ===
  Raw score      : -1067.687828
  PnL component  : -787.265328
  Sharpe comp.   : 0.332274
  Drawdown comp. : 236.922111
  Fee drag comp. : 40.616304
  Inventory comp.: 3.216359
  Trade penalty  : 0.000000
  Rejected?      : False


# 10. Walk-Forward Validation

Set up a `TimeSeriesCV` splitter to perform walk-forward cross-validation.
Each fold trains on a fixed-length window and tests on the subsequent out-of-sample
period.  Per-fold metrics are collected and displayed.

In [11]:
try:
    from pmm_lab.objective.walkforward import run_walk_forward

    if candles is None or baseline_config is None:
        raise ValueError("No candle data or config available.")

    wf_result = run_walk_forward(
        candles=candles,
        config=baseline_config,
        pair_rules=pair_rules,
        bar_interval_seconds=BAR_INTERVAL_SECONDS,
        dataset_hash=DATASET_HASH,
        train_days=42.0,
        test_days=14.0,
        step_days=14.0,
    )

    print(f"Walk-forward: {len(wf_result.folds)} folds, aggregate score: {wf_result.aggregate_score:.4f}")
    print()
    rows = []
    for fr in wf_result.folds:
        rows.append({
            "Fold": fr.fold_index,
            "PnL %": f"{fr.test_metrics.pnl_pct:.2f}",
            "Sharpe": f"{fr.test_metrics.sharpe:.2f}",
            "MaxDD %": f"{fr.test_metrics.max_drawdown_pct:.2f}",
            "Trades": fr.test_metrics.trade_count,
            "Objective": f"{fr.test_objective.raw_score:.4f}",
        })
    display(pd.DataFrame(rows))
except Exception as exc:
    print(f"[SKIP] Walk-forward failed: {exc}")
    wf_result = None

Walk-forward: 10 folds, aggregate score: -85.5976



,Fold,PnL %,Sharpe,MaxDD %,Trades,Objective
0,0,-105.23,4.91,106.85,685,-149.9707
1,1,-71.86,-18.28,72.17,697,-101.9468
2,2,-53.35,-14.56,54.73,769,-78.5387
3,3,-52.01,-17.09,52.55,560,-76.2794
4,4,-40.08,-13.68,41.38,484,-59.7342
5,5,-44.31,-12.16,45.62,701,-66.3648
6,6,-63.67,-18.17,65.33,575,-92.2444
7,7,-20.89,-6.91,23.45,623,-35.1919
8,8,-52.20,-12.80,52.59,653,-76.4015
9,9,-76.06,-20.71,76.09,676,-108.1844


# 11. Stress Testing

Run the baseline configuration against a set of stress-test scenarios
(e.g., high-volatility regime, low-liquidity regime, trending market).
Display per-scenario metrics in a comparison table.

In [12]:
try:
    from pmm_lab.objective.stress import run_stress_tests

    if candles is None or baseline_config is None:
        raise ValueError("No candle data or config available.")

    stress_report = run_stress_tests(
        candles=candles,
        config=baseline_config,
        pair_rules=pair_rules,
        bar_interval_seconds=BAR_INTERVAL_SECONDS,
    )

    print(f"Stress tests: {len(stress_report.scenario_results)} scenarios")
    print(f"Worst scenario: {stress_report.worst_scenario} (score: {stress_report.worst_score:.4f})")
    print()
    rows = []
    for sr in stress_report.scenario_results:
        rows.append({
            "Scenario": sr.scenario.name,
            "PnL %": f"{sr.metrics.pnl_pct:.2f}",
            "Sharpe": f"{sr.metrics.sharpe:.2f}",
            "MaxDD %": f"{sr.metrics.max_drawdown_pct:.2f}",
            "Trades": sr.metrics.trade_count,
            "Objective": f"{sr.objective.raw_score:.4f}",
        })
    display(pd.DataFrame(rows))
except Exception as exc:
    print(f"[SKIP] Stress testing failed: {exc}")
    stress_report = None

Stress tests: 10 scenarios
Worst scenario: fees_2x (score: -1373.2462)



,Scenario,PnL %,Sharpe,MaxDD %,Trades,Objective
0,fees_1.5x,-888.81,1.48,890.96,12796,-1221.1078
1,fees_2x,-990.35,-0.51,992.17,12796,-1373.2462
2,latency_plus1,-796.81,-1.39,799.28,12720,-1080.5759
3,latency_plus2,-806.56,1.97,809.37,12548,-1090.7624
4,latency_plus3,-778.66,3.84,781.33,12202,-1054.8309
5,low_liquidity,-743.97,1.82,747.66,19571,-1008.0005
6,very_low_liquidity,-712.04,-1.11,715.09,29638,-961.6971
7,high_slippage,-807.60,1.43,810.05,12796,-1093.4797
8,extreme_slippage,-848.28,0.11,850.66,12796,-1149.8558
9,combined_adverse,-854.21,-1.37,856.56,19483,-1169.1779


# 12. Optuna Study Setup and Optimization

Create an Optuna study (optionally backed by Postgres via `OPTUNA_STORAGE`),
define the objective function, and run the hyper-Bayesian optimization loop.
The number of trials is configurable.

In [13]:
# --- Optimization configuration ---
N_TRIALS = 1000
STUDY_NAME = f"{CONNECTOR}_{TRADING_PAIR}_{INTERVAL}_pmm_dynamic_v1"
# -----------------------------------
print(f"Study name: {STUDY_NAME}")
print(f"Trials: {N_TRIALS}")

Study name: nonkyc_SAL-USDT_5m_pmm_dynamic_v1
Trials: 1000


In [14]:
try:
    from pmm_lab.optuna.study import create_study, run_optimization
    from pmm_lab.optuna.objective_wrapper import create_objective
    from pmm_lab.optuna.callbacks import DegeneracyCheckCallback, TrialLoggingCallback

    if candles is None:
        raise ValueError("No candle data available.")

    ref_price = float(np.median(candles["close"]))

    study = create_study(
        study_name=STUDY_NAME,
        storage_url=OPTUNA_STORAGE if OPTUNA_STORAGE else None,
    )

    objective_fn = create_objective(
        candles=candles,
        pair_rules=pair_rules,
        bar_interval_seconds=BAR_INTERVAL_SECONDS,
        dataset_hash=DATASET_HASH,
        reference_price=ref_price,
        train_days=42.0,
        test_days=14.0,
        step_days=14.0,
        run_stress=False,  # set True for full robustness (slower)
    )

    run_optimization(
        study, objective_fn, n_trials=N_TRIALS,
        callbacks=[TrialLoggingCallback(log_every=5), DegeneracyCheckCallback()],
    )

    n_complete = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    n_pruned = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    print(f"\nOptimization complete. Completed: {n_complete}, Pruned: {n_pruned}")
    print(f"Best value: {study.best_value:.6f}")
except Exception as exc:
    print(f"[SKIP] Optuna optimization failed: {exc}")
    study = None

/quants-lab/research_notebooks/market_lab/pmm_dynamic/pmm_lab/optuna/study.py:33: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
[I 2026-03-06 15:42:53,200] Using an existing study with name 'nonkyc_SAL-USDT_5m_pmm_dynamic_v1' instead of creating a new one.
[W 2026-03-06 15:42:53,729] The parameter `macd_slow` in Trial#600 is sampled independently using `RandomSampler` instead of `TPESampler`, potentially degrading the optimization performance. This fallback happend because dynamic search space is not supported for `multivariate=True`. You can suppress this warning by setting `warn_independent_sampling` to `False` in the constructor of `TPESampler` if this independent sampling is intended behavior.
[I 2026-03-06 15:42:53,935] Trial 600 pruned. 
[W 2026-03-06 15:42:54,177] The parameter `macd_slow` in Trial#601 is sampled independently using `RandomSampler` instead of `TPESampler`


Optimization complete. Completed: 1112, Pruned: 488
Best value: 808.384076


# 13. Best-Trial Review and Top-N Comparison

Display the best trial's parameters and objective value, then show a comparison
table of the top 10 trials ranked by objective.

In [15]:
try:
    if study is None:
        raise ValueError("No Optuna study available.")

    best = study.best_trial
    print("=== Best Trial ===")
    print(f"  Trial #   : {best.number}")
    print(f"  Objective : {best.value:.6f}")
    print(f"  Params    :")
    for k, v in sorted(best.params.items()):
        print(f"    {k:30s} = {v}")

    # Top-N comparison
    TOP_N = 10
    complete_trials = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    sorted_trials = sorted(complete_trials, key=lambda t: t.value, reverse=True)[:TOP_N]
    rows = []
    for t in sorted_trials:
        rows.append({
            "Trial": t.number,
            "Objective": f"{t.value:.4f}",
            **{k: t.params.get(k, "") for k in ["buy_n_levels", "sell_n_levels", "stop_loss", "take_profit"]},
        })
    print(f"\n=== Top {TOP_N} Trials ===")
    display(pd.DataFrame(rows))
except Exception as exc:
    print(f"[SKIP] Best-trial review failed: {exc}")

=== Best Trial ===
  Trial #   : 1571
  Objective : 808.384076
  Params    :
    amount_skew                    = 2.9231975763052906
    buy_n_levels                   = 2
    buy_side_weight                = 0.7953200971336091
    buy_spread_base                = 0.2666436702529785
    buy_spread_ratio               = 1.2728064292621248
    cooldown_time                  = 205.1292671147308
    executor_refresh_time          = 377.0066781008875
    macd_fast                      = 8
    macd_signal                    = 15
    macd_slow                      = 84
    natr_length                    = 41
    sell_n_levels                  = 1
    sell_spread_base               = 2.1523807174134966
    sell_spread_ratio              = 1.703535807803702
    stop_loss                      = 0.17417519112110125
    take_profit                    = 0.0690823002970069
    time_limit                     = 64781
    total_amount_quote             = 25.28366790695684
    trailing_stop_activation  

,Trial,Objective,buy_n_levels,sell_n_levels,stop_loss,take_profit
0,1571,808.3841,2,1,0.174175,0.069082
1,1555,710.4151,2,3,0.171676,0.070912
2,1575,596.8561,2,4,0.172290,0.065348
3,1484,554.9799,2,7,0.167353,0.083979
4,1596,534.4109,3,5,0.176926,0.069995
5,1135,523.6675,3,4,0.167649,0.081909
6,1551,513.0469,3,3,0.169365,0.074848
7,1237,496.7374,2,2,0.118628,0.076562
8,1330,479.4417,2,2,0.148201,0.079632
9,1188,472.6653,2,1,0.149804,0.079283


# 14. Hummingbot YAML Export and Validation

Export the best trial's parameters as a Hummingbot-compatible YAML configuration
file and run schema validation to ensure it is well-formed.

In [16]:
try:
    from pmm_lab.export.hb_yaml import export_yaml, ExportParams
    from pmm_lab.export.validate_export import validate_yaml_file
    from pmm_lab.optuna.canonicalizer import canonicalize_params

    if study is None:
        raise ValueError("No Optuna study available.")

    # Reconstruct SimConfig from best trial params
    ref_price = float(np.median(candles["close"]))
    best_config, reject = canonicalize_params(study.best_trial.params, pair_rules, ref_price)
    if best_config is None:
        raise ValueError(f"Best trial params rejected: {reject}")

    export_params = ExportParams(
        connector_name=CONNECTOR,
        trading_pair=TRADING_PAIR,
        candles_connector=CONNECTOR,
        candles_trading_pair=TRADING_PAIR,
        interval=INTERVAL,
    )

    yaml_path = export_yaml(
        config=best_config,
        output_path=f"artifacts/{STUDY_NAME}_best.yml",
        export_params=export_params,
        metadata={"dataset_hash": DATASET_HASH, "trial": study.best_trial.number,
                  "objective": study.best_value},
    )
    print(f"YAML exported to: {yaml_path}")

    validation = validate_yaml_file(yaml_path)
    print(f"Validation: {'PASS' if validation.valid else 'FAIL'} (mode: {validation.mode})")
    if not validation.valid:
        for err in validation.errors:
            print(f"  ERROR: {err}")
except Exception as exc:
    print(f"[SKIP] YAML export/validation failed: {exc}")

YAML exported to: artifacts/nonkyc_SAL-USDT_5m_pmm_dynamic_v1_best.yaml
Validation: PASS (mode: mirror)


# 15. Markdown Report Generation

Generate a comprehensive Markdown report summarizing the optimization run,
including dataset info, baseline metrics, best parameters, walk-forward results,
and stress-test outcomes.

In [17]:
try:
    from pmm_lab.report.report_md import generate_report, run_stop_ship_checks

    if study is None:
        raise ValueError("No study available.")

    # Build dataset summary
    dataset_summary = {
        "connector": CONNECTOR,
        "trading_pair": TRADING_PAIR,
        "interval": INTERVAL,
        "n_candles": len(candles) if candles is not None else 0,
        "dataset_hash": DATASET_HASH,
    }

    # Get best trial metrics from user attrs (if available) or recompute
    best_params_dict = study.best_trial.params

    # Run stop-ship checks
    checks = run_stop_ship_checks(
        best_metrics=metrics,
        best_objective=obj,
        walkforward_result=wf_result if 'wf_result' in dir() and wf_result else None,
        stress_report=stress_report if 'stress_report' in dir() and stress_report else None,
        dataset_hash=DATASET_HASH,
    )

    report_path = generate_report(
        study_name=STUDY_NAME,
        dataset_summary=dataset_summary,
        best_params=best_params_dict,
        best_metrics=metrics,
        best_objective=obj,
        walkforward_result=wf_result if 'wf_result' in dir() and wf_result else None,
        stress_report=stress_report if 'stress_report' in dir() and stress_report else None,
        stop_ship_checks=checks,
        output_path=f"artifacts/{STUDY_NAME}_report.md",
    )
    print(f"Report saved to: {report_path}")
except Exception as exc:
    print(f"[SKIP] Report generation failed: {exc}")

Report saved to: # PMM Dynamic Optimization Report: nonkyc_SAL-USDT_5m_pmm_dynamic_v1

Generated: 2026-03-06 23:02:39 UTC

## Dataset Summary

- **connector**: nonkyc
- **trading_pair**: SAL-USDT
- **interval**: 5m
- **n_candles**: 53143
- **dataset_hash**: c3f47aaeb70a0c77701fe169f11f46a44df040ab96cbd4741646bf7b6d5c3bcf

## Best Parameters

| Parameter | Value |
|-----------|-------|
| amount_skew | 2.9231975763052906 |
| buy_n_levels | 2 |
| buy_side_weight | 0.7953200971336091 |
| buy_spread_base | 0.2666436702529785 |
| buy_spread_ratio | 1.2728064292621248 |
| cooldown_time | 205.1292671147308 |
| executor_refresh_time | 377.0066781008875 |
| macd_fast | 8 |
| macd_signal | 15 |
| macd_slow | 84 |
| natr_length | 41 |
| sell_n_levels | 1 |
| sell_spread_base | 2.1523807174134966 |
| sell_spread_ratio | 1.703535807803702 |
| stop_loss | 0.17417519112110125 |
| take_profit | 0.0690823002970069 |
| time_limit | 64781 |
| total_amount_quote | 25.28366790695684 |
| trailing_stop_activa

In [5]:
import optuna

studies = optuna.study.get_all_study_names(storage=OPTUNA_STORAGE)
for s in studies:
    print(s)

mexc_BTC-USDT_1m_pmm_dynamic_v2
